# Lab 2a — First Regression Model
**SDA-AIE-111 · Day 1 · Hour 5 · 50 minutes · pairs**

**Objective.** Beat the Lab 1 dummy baseline with interpretable regression models for basket value; read residuals like a diagnostic; meet regularisation.

The target is `avg_basket_sar` — the customer's average basket value, which the CRM team uses to size retention vouchers. Because it is the target, it is **excluded from the features** (predicting a number from itself is not modelling).


In [ ]:
# --- Course setup (identical in every lab) -----------------------------------
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_repo_root() -> Path:
    """Locate the course repo root by looking for data/manafeth_customers.parquet."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "manafeth_customers.parquet").exists():
            return cand
    raise FileNotFoundError(
        "Course data not found — copy the Manafeth data package files into <repo>/data/")

ROOT = find_repo_root()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)

In [ ]:
from sklearn.model_selection import train_test_split

customers = pd.read_parquet(DATA / "manafeth_customers.parquet")
LEAKY = ["refund_issued", "support_ticket_after_snapshot", "next_month_orders"]

# Same split discipline as Lab 1 (same seed -> same rows)
X_all = customers.drop(columns=["churned_30d", "customer_id", *LEAKY])
y_all = customers["churned_30d"]
X_train, X_test, y_cls_train, y_cls_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE)

# Regression target and numeric-only first pass (categoricals join in Module 3)
TARGET_REG = "avg_basket_sar"
num_cols = ["orders_per_month", "days_since_last_order", "tenure_months",
            "distinct_categories", "promo_usage_rate"]

y_basket = X_train[TARGET_REG]
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train[num_cols], y_basket, test_size=0.25, random_state=RANDOM_STATE)
print(X_tr.shape, X_val.shape)

## Task 1 — Linear regression vs the dummy *(10 min)*

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor

# TODO: fit DummyRegressor(strategy="mean") and LinearRegression on X_tr
# TODO: report validation MAE for both (in SAR) and R^2 for the linear model
# TODO: print the coefficients sorted by absolute size, with units


## Task 2 — Residuals: the model's confession *(15 min)*

Plot residuals vs predictions. A funnel (errors growing with basket size) says: try a log target.


In [ ]:
# TODO: scatter-plot residuals (y_val - pred) against predictions; add a zero line
# TODO: refit on np.log1p(y_tr), back-transform predictions with np.expm1,
#       and compare MAE in SAR against the plain model


## Task 3 — Ridge and lasso *(15 min)*

Regularisation is the first overfitting control. Sweep alpha logarithmically **on scaled features** and watch which lasso coefficients hit zero.


In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# TODO: sweep alpha in {0.01, 0.1, 1, 10, 100} for Ridge and Lasso
#       (scale features first — put StandardScaler and the model in a Pipeline)
# TODO: tabulate validation MAE per (model, alpha)
# TODO: which lasso coefficients hit zero at alpha=10? Sanity-check against domain sense.


## Task 4 — Three-sentence summary *(10 min)*

Write it in this cell: best model, MAE in SAR, one surprising coefficient.

> …

**Commit:** `feat(lab2a): linear regression beats dummy; residual diagnosis`
